In [1]:
# ==========================================
# CHECK GPU
# ==========================================
!nvidia-smi

Tue Jun  9 14:22:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**If you develop an chatbot this would be the architecture**




Initial Phase (Document Ingestion)

Documents (PDF, DOCX, TXT, etc.)
        ↓
LangChain
        ↓
Text Splitter (Chunking)
        ↓
Embedding Model
(OpenAI / Gemini / Hugging Face)
        ↓
Embeddings (Vectors)
        ↓
Vector Database
(Chroma / Pinecone / FAISS)






User Query Phase

User Question
        ↓
LangChain
        ↓
Embedding Model
(OpenAI / Gemini / Hugging Face)
        ↓
Question Embedding
        ↓
Vector Database
(Similarity Search)
        ↓
Retrieve Relevant Chunks
        ↓
GPT / Gemini / LLM
        ↓
Generate Natural Language Response
        ↓
User


In [2]:
# ==========================================
# BLOCK 1: INSTALL PACKAGES
# ==========================================
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-chroma
!pip install langchain-groq
!pip install langchain-classic
!pip install pypdf
!pip install sentence-transformers
!pip install gradio
!pip install langsmith

In [3]:




# ==========================================
# BLOCK 2: ENVIRONMENT SETUP
# Set ALL keys BEFORE any imports
# ==========================================
import os

# Groq API Key
os.environ["GROQ_API_KEY"] = "add api"

# LangSmith / LangChain Keys
os.environ["LANGCHAIN_API_KEY"] = "add api"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "RAG_LLMOPS_PROJECT"

# Verify
print("✅ GROQ_API_KEY:", "Set" if os.getenv("GROQ_API_KEY") else "❌ Missing")
print("✅ LANGCHAIN_API_KEY:", "Set" if os.getenv("LANGCHAIN_API_KEY") else "❌ Missing")
print("✅ LANGCHAIN_TRACING_V2:", os.getenv("LANGCHAIN_TRACING_V2"))
print("✅ LANGCHAIN_ENDPOINT:", os.getenv("LANGCHAIN_ENDPOINT"))
print("✅ LANGCHAIN_PROJECT:", os.getenv("LANGCHAIN_PROJECT"))


✅ GROQ_API_KEY: Set
✅ LANGCHAIN_API_KEY: Set
✅ LANGCHAIN_TRACING_V2: true
✅ LANGCHAIN_ENDPOINT: https://api.smith.langchain.com
✅ LANGCHAIN_PROJECT: RAG_LLMOPS_PROJECT


In [5]:
# ==========================================
# BLOCK 3: VERIFY LANGSMITH CONNECTION
# ==========================================

from langsmith import Client

client = Client()

print("LangSmith Connected Successfully")

LangSmith Connected Successfully


In [6]:
# ==========================================
# BLOCK 4: IMPORT REQUIRED LIBRARIES
# ==========================================

import requests

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [9]:
# ==========================================
# BLOCK 5: DOWNLOAD PDF DOCUMENT
# ==========================================

url = "https://dspmuranchi.ac.in/pdf/Blog/Python%20Built-In%20Functions.pdf"

response = requests.get(url)

with open("python_built_in_functions.pdf", "wb") as f:
    f.write(response.content)

print("PDF Downloaded Successfully")

PDF Downloaded Successfully


In [10]:
# ==========================================
# BLOCK 6: LOAD PDF DOCUMENT
# ==========================================

# Load PDF file into LangChain Documents

loader = PyPDFLoader(
    "python_built_in_functions.pdf"
)

documents = loader.load()

print("Total Pages:", len(documents))

print(documents[0])

Total Pages: 15
page_content='Python Built-In Functions 
 
Gaurav Kr. suman       MIT5' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2020-06-04T12:32:44+05:30', 'title': 'Python Built-In Functions', 'author': 'Gaurav Kr. suman', 'moddate': '2020-06-04T12:32:44+05:30', 'source': 'python_built_in_functions.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [11]:
# ==========================================
# BLOCK 7: SPLIT DOCUMENT INTO CHUNKS
# ==========================================

# Split large document into smaller chunks

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

texts = text_splitter.split_documents(documents)

print("Total Chunks:", len(texts))

Total Chunks: 15


In [12]:
# ==========================================
# BLOCK 8: LOAD EMBEDDING MODEL
# ==========================================

# Convert text into vector embeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

/tmp/ipykernel_23824/3640802991.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully


In [13]:
# ==========================================
# BLOCK 9: CREATE VECTOR DATABASE (CHROMADB)
# ==========================================

# Store embeddings in Chroma Vector Database

vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="vector_db"
)

print("Vector Database Created Successfully")

Vector Database Created Successfully


In [14]:
# ==========================================
# BLOCK 10: CREATE RETRIEVER
# ==========================================

# Retriever performs similarity search

retriever = vectordb.as_retriever(
    search_kwargs={"k":3}
)

print("Retriever Created Successfully")

Retriever Created Successfully


In [15]:
# ==========================================
# BLOCK 11: LOAD GROQ LLM
# ==========================================

# Load Llama 3.3 Model from Groq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


In [16]:
# ==========================================
# BLOCK 12: BUILD RETRIEVAL QA CHAIN
# ==========================================

# Combine Retriever + LLM

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RetrievalQA Chain Created Successfully")

RetrievalQA Chain Created Successfully


In [17]:
# ==========================================
# BLOCK 13: TEST RAG PIPELINE
# ==========================================

query = "What is lambda function?"

response = qa_chain.invoke(
    {"query": query}
)

print(response["result"])

A lambda function is a small, anonymous function in Python that can take any number of arguments, but can only have one expression. It is defined using the `lambda` keyword and is often used in combination with other functions such as `filter()`, `map()`, and `reduce()`. 

In the context provided, a lambda function is used with the `filter()` function to filter out items from a list for which the condition is True. For example: 
```
>>> list(filter(lambda x: x % 2 == 0, [1, 2, 0, False]))
[2, 0, False]
```
This lambda function takes an argument `x` and returns `True` if `x` is even (i.e., `x % 2 == 0`), and `False` otherwise. The `filter()` function then uses this lambda function to filter out the items from the list that are not even.


In [24]:
# ==========================================
# BLOCK 14: CREATE GRADIO CHATBOT UI
# ==========================================

# import gradio as gr

# # Function for answering user questions

# def chatbot(question):

#     response = qa_chain.invoke(
#         {"query": question}
#     )

#     return response["result"]



import gradio as gr

def chatbot_response(question):

    try:

        response = qa_chain.invoke(
            {"query": question}
        )

        answer = response["result"]

        sources = []

        for doc in response["source_documents"]:

            if "source" in doc.metadata:
                sources.append(doc.metadata["source"])

        source_text = "\n".join(
            list(set(sources))
        )

        final_response = f"""
{answer}

---------------------------------------
📄 Source:
{source_text}
"""

        return final_response

    except Exception as e:

        return f"Error: {str(e)}"

In [ ]:
# ==========================================
# BLOCK 15: LAUNCH APPLICATION
# ==========================================

# demo = gr.Interface(
#     fn=chatbot,
#     inputs="text",
#     outputs="text",
#     title="RAG PDF Chatbot",
#     description="LangChain + ChromaDB + Groq + LangSmith + Gradio"
# )

# demo.launch(share=True)


demo = gr.Interface(
    fn=chatbot_response,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask questions from your PDF...",
        label="Your Question"
    ),

    outputs=gr.Textbox(
        lines=12,
        label="Output"
    ),

    title="📚 RAG PDF Chatbot",

    description="""
LLMOps Project using:

• LangChain
• ChromaDB
• Hugging Face Embeddings
• Groq Llama 3.3
• LangSmith Tracing
• Gradio UI
""",

    examples=[
        ["What is abs() function?"],
        ["Explain lambda function"],
        ["What is filter() function?"],
        ["What is map() function?"],
        ["Summarize the PDF"]
    ],

    theme=gr.themes.Soft()
)

demo.launch(
    share=True,
    debug=True
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://436a00c06d0282eaec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
response = qa_chain.invoke(
    {"query":"What is lambda function?"}
)

print(response["result"])

A lambda function is a small, anonymous function in Python that can take any number of arguments, but can only have one expression. It is defined using the `lambda` keyword and is often used in combination with other functions such as `filter()`, `map()`, and `reduce()`. 

In the context provided, it is used with the `filter()` function to filter out items from a list for which the condition is True. For example: 
```
list(filter(lambda x: x % 2 == 0, [1, 2, 0, False]))
```
This will return a list of items from the original list for which the condition `x % 2 == 0` is True, i.e., the even numbers.


In [21]:
import os

print(os.getenv("LANGCHAIN_PROJECT"))

RAG_LLMOPS_PROJECT


In [22]:
from langsmith import traceable

@traceable
def test_trace():
    return "Hello LangSmith"

print(test_trace())

Hello LangSmith


In [23]:
from langsmith import Client

client = Client()

for p in client.list_projects():
    print(p.name)

RAG_LLMOPS_PROJECT
